# Data Cleaning & Reporting Automation

**Goal:** Automate data cleaning and reporting workflows using Python. Handle missing values, duplicates, and inconsistent data automatically, and generate automated reports with visual summaries.

**Approach:** A reusable `DataCleaningAutomation` class (in `data_cleaning_automation.py`) is built that works on *any* similarly-shaped CSV — not hardcoded to one dataset. It is demonstrated here on a deliberately 'messy' raw version of the Car Sales dataset (missing values, duplicate rows, inconsistent text casing/whitespace, mixed date formats, and invalid numeric entries were injected to simulate realistic raw data — the original dataset was already clean).

## 1. Generate a Realistic 'Messy' Raw Dataset

To genuinely demonstrate automated cleaning, common real-world data problems are injected into a copy of the clean Car Sales dataset: missing values, exact + near duplicate rows (different casing), inconsistent text (whitespace/casing), mixed date formats, and invalid numeric entries.

In [ ]:
# See make_messy_data.py for the full generation logic
import pandas as pd
messy = pd.read_csv('Car_Sales_RAW_messy.csv')
print(messy.shape)
messy.head()

## 2. Import the Automation Pipeline

In [ ]:
from data_cleaning_automation import DataCleaningAutomation

pipeline = DataCleaningAutomation('Car_Sales_RAW_messy.csv', id_column='Sale ID')

## 3. Run the Full Automated Cleaning Pipeline

A single `.run()` call performs, in order:
1. Load data
2. Profile raw data quality (missing %, duplicates, inconsistent text)
3. Remove duplicates, standardize text formatting, parse dates consistently
4. Re-check for duplicates created by text standardization
5. Flag and fix invalid numeric entries (e.g. negative discount, zero units)
6. Impute missing values (median for numeric, mode for categorical)
7. Recompute dependent columns (Revenue) for consistency
8. Profile cleaned data quality

In [ ]:
pipeline.run(
    numeric_validity_rules={
        'Units Sold': 'positive',
        'Discount (%)': 'non_negative',
        'Unit Price (INR)': 'positive',
    },
    formula_cols={
        'Revenue (INR)': lambda df: df['Units Sold'] * df['Unit Price (INR)'] *
                                     (1 - df['Discount (%)'] / 100)
    }
)

## 4. Generate the Automated Report (Charts + Logs + Cleaned Data)

In [ ]:
summary = pipeline.save_report(output_dir='.')
summary

## 5. Visual Summary — Before vs After

In [ ]:
from IPython.display import Image, display
display(Image('chart_quality_before_after.png'))
display(Image('chart_quality_score.png'))
display(Image('chart_missing_by_column.png'))

## 6. Audit Trail

Every cleaning action is logged in order, so the pipeline is fully transparent and reproducible.

In [ ]:
with open('cleaning_audit_log.txt') as f:
    print(f.read())

## 7. Cleaned Data Preview

In [ ]:
cleaned = pd.read_csv('cleaned_data.csv')
print(cleaned.shape)
cleaned.head()

## 8. Reusability

Because the pipeline auto-detects column types (numeric vs categorical vs date) rather than hardcoding column names, the same `DataCleaningAutomation` class can be pointed at a different dataset (e.g. the Stationery Sales dataset) with only the validity rules changed — no rewrite needed. This is what makes it a genuine **automation tool** rather than a one-off script.